<a href="https://www.kaggle.com/code/leonardoterra/ai-usage-vs-gpa-improvement?scriptVersionId=354416746" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

<a href="https://www.kaggle.com/code/leonardoterra" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

<div style="text-align:center; border-radius:15px; padding:15px; color:white; margin:0; font-family: 'Impact', sans-serif; background: #2E0249; background: #11001C; box-shadow: 0px 4px 8px rgba(0, 0, 0, 0.3); overflow:hidden; margin-bottom: 1em; letter-spacing: 1px">
  <div style="font-size:150%; color:#FFFFFF"><b>AI USAGE VS GPA IMPROVEMENT</b></div>
</div>

This analysis is based on a dataset extracted from Kaggle.com. This dataset presents data related to student AI tool usage in 2026, including GPA before and after AI adoption, primary tools used, main usage cases, time saved, and ethics concerns. The goal of this analysis is to evaluate whether AI usage has a meaningful impact on academic performance and to explore behavioral patterns across different student majors.

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+E
# nter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')

In [ ]:
'''
Run on Kaggle or Google Colab
#df = pd.read_csv('/kaggle/input/datasets/sohaibdevv/ai-and-student-life-2026-the-new-normal/AI_Impact_Student_Life_2026.csv')
'''

#Run it locally
import kagglehub
kagglehub.login()

from pathlib import Path
path = kagglehub.dataset_download("sohaibdevv/ai-and-student-life-2026-the-new-normal")
csv_path = Path(path) / "AI_Impact_Student_Life_2026.csv"

df = pd.read_csv(csv_path)
df

## **Data Exploration**

    During this step we can explore the shape of the data we are working with. It's usual to check and deal with the number of columns and rows, missing values, outliers and other inconsistencies that might appear.

In [ ]:
df.info()

In [ ]:
data_assessment = pd.DataFrame({
    "Column": df.columns,
    "Data Type": [df[col].dtype for col in df.columns],
    "Missing Values": [f"{df[col].isna().mean() * 100:.1f}%" for col in df.columns],
    "Unique Values": [df[col].nunique() for col in df.columns],
    "Example Value": [df[col].dropna().iloc[0] if df[col].notna().any() else None for col in df.columns]
})
data_assessment

In [ ]:
feature = {
    "Student_ID": "A unique identifier for each student.",
    "Age": "Age of the student, ranging from 18 to 25 years.",
    "Major": "The student's field of study (e.g., Software Engineering, Business, Arts).",
    "Primary_AI_Tool": "The main AI platform the student uses for academic work.",
    "Task_Frequency_Daily": "Estimated number of times a student interacts with AI tools per day.",
    "Main_Usage_Case": "The primary way AI is utilized (e.g., Debugging, Research, Exam Prep).",
    "GPA_Baseline": "The student's Grade Point Average before integrating AI into their study routine.",
    "GPA_Post_AI": "The student's current Grade Point Average after adopting AI tools.",
    "Time_Saved_Hours_Weekly": "Estimated hours saved per week by using AI for manual or repetitive tasks.",
    "AI_Ethics_Concern": "Level of concern regarding academic integrity and original work (Low/Medium/High)."
}

In [ ]:
# GPA Improvement Delta
df['Improvement %'] = (df['GPA_Post_AI'] / df['GPA_Baseline']-1).round(2)

## **Descriptive Statistics**

    This step seeks to understand the interaction and correlation among the features. Here it's possible to check multicolinearity, plot visulizations and highlight key points.

#### **Feature Distribution**

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(8, 3))

ax[0] = sns.histplot(data=df, x='GPA_Baseline', ax=ax[0], kde=True)
ax[1] = sns.histplot(data=df, x='GPA_Post_AI', ax=ax[1], kde=True)

sns.despine(left=False, right=True, top=True, bottom=False)
plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=3, figsize=(12, 3))

ax[0] = sns.boxplot(data=df, x='GPA_Baseline', ax=ax[0])
ax[1] = sns.boxplot(data=df, x='GPA_Post_AI', ax=ax[1])
ax[2] = sns.boxplot(data=df, x='Improvement %', ax=ax[2])

sns.despine(left=False, right=True, top=True, bottom=False)
plt.tight_layout()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

cols = df.drop(['Student_ID','Age','Task_Frequency_Daily','Task_Frequency_Daily','GPA_Baseline','GPA_Post_AI','Time_Saved_Hours_Weekly','Career_Confidence_Score'], axis=1).columns.to_list()

fig, axes = plt.subplots(1, 4, figsize=(35, 8))

for ax, col in zip(axes.flatten(), cols):
    sns.countplot(data=df, y=col, ax=ax)
    ax.set_title(col)

sns.despine(left=False, right=True, top=True, bottom=False)
plt.tight_layout()

    Github Copilot is the most used AI overall followed by Gemini Pro and Perplexity.
    The main usage of the models goes to Exam Prep, followed by Literature Review and Essay Drafting.

In [ ]:
# Baseline GPA x After AI GPA

df[['GPA_Baseline', 'GPA_Post_AI']].mean().round(2)

In [ ]:
corr = df.select_dtypes(include='number').corr()
mask = np.triu(np.ones_like(corr, dtype=bool)) # Mask to remove the upper side and facilitate visualization

plt.figure(figsize=(10,6))
sns.heatmap(corr, mask=mask, annot=True, cmap='coolwarm', center=0)

sns.despine(left=False, right=True, top=True, bottom=False)
plt.title("Correlation Analysis", fontsize=9)
plt.tight_layout()

#### **Paired Distributions**
    Checking Majors against many features to see how each field performs before and after AI, as well as how they use it.

##### **Primary AI tool by Major**

In [ ]:
# Dataframe
major_tool = (df.groupby('Major')['Primary_AI_Tool'].value_counts(normalize=True).round(3)*100).reset_index()
major_tool.rename(columns={'proportion':'share'}, inplace=True)

# Plot
axes = sns.catplot(data=major_tool, x='Primary_AI_Tool', y='share', col='Major', hue='Primary_AI_Tool', col_wrap=3, kind='bar', height=3, aspect=1.5, legend=True)
axes.set_axis_labels('', 'Share (%)')
axes.set_titles(col_template='{col_name}')

for ax in axes.axes.flat:
    ax.set_xticklabels([])
    ax.tick_params(axis='x', length=0)
    for bar in ax.patches:
        height = bar.get_height()
        if height > 0:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                height + 0.3,
                f'{height:.1f}%',
                ha='center',
                va='bottom',
                fontsize=9
            )

sns.move_legend(axes, loc='upper right', bbox_to_anchor=(1.15, 0.95))
sns.despine(left=False, right=True, top=True, bottom=False)
plt.tight_layout()

##### **Main use case by Major**

In [ ]:
# Dataframe
major_use_case = (df.groupby('Major')['Main_Usage_Case'].value_counts(normalize=True).round(3)*100).reset_index()
major_use_case.rename(columns={'proportion':'share'}, inplace=True)

# Plot
axes = sns.catplot(data=major_use_case, x='Main_Usage_Case', y='share', col='Major', hue='Main_Usage_Case', col_wrap=3, kind='bar', height=3, aspect=1.5, legend=True)
axes.set_axis_labels('', 'Share (%)')
axes.set_titles(col_template='{col_name}')

for ax in axes.axes.flat:
    ax.set_xticklabels([])
    ax.tick_params(axis='x', length=0)
    for bar in ax.patches:
        height = bar.get_height()
        if height > 0:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                height + 0.3,
                f'{height:.1f}%',
                ha='center',
                va='bottom',
                fontsize=9
            )

sns.move_legend(axes, loc='upper right', bbox_to_anchor=(1.15, 0.95))
sns.despine(left=False, right=True, top=True, bottom=False)
plt.tight_layout()

##### **Ethics concern by Major**

In [ ]:
# Dataframe
major_ethics = (df.groupby('Major')['AI_Ethics_Concern'].value_counts(normalize=True).round(3)*100).reset_index()
major_ethics.rename(columns={'proportion':'share'}, inplace=True)

# Plot
axes = sns.catplot(data=major_ethics, x='AI_Ethics_Concern', y='share', col='Major', hue='AI_Ethics_Concern', col_wrap=3, kind='bar', height=3, aspect=1.5, legend=True)
axes.set_axis_labels('', 'Share (%)')
axes.set_titles(col_template='{col_name}')

for ax in axes.axes.flat:
    ax.set_xticklabels([])
    ax.tick_params(axis='x', length=0)
    for bar in ax.patches:
        height = bar.get_height()
        if height > 0:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                height + 0.3,
                f'{height:.1f}%',
                ha='center',
                va='bottom',
                fontsize=9
            )

sns.move_legend(axes, loc='upper right', bbox_to_anchor=(1.15, 0.95))
sns.despine(left=False, right=True, top=True, bottom=False)
plt.tight_layout()

##### **GPA improvement After AI**

In [ ]:
# Dataframe

major_gpa = (df.groupby('Major')['Improvement %'].mean().round(3)*100).reset_index()
order = major_gpa.sort_values('Improvement %', ascending=False)['Major'].tolist()

# Plot
fig, ax = plt.subplots(1, 1, figsize=(10, 4))
sns.barplot(data=major_gpa, x='Major', y='Improvement %', order=order)
ax.set_title('GPA Improvement % by Major')
ax.set_xlabel('')

ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha='right')
for bar in ax.patches:
    height = bar.get_height()
    if height > 0:
        ax.text(bar.get_x() + bar.get_width() / 2, height + 0.02, f'{height:.2f}%', ha='center', va='bottom', fontsize=9)

sns.despine(left=False, right=True, top=True, bottom=False)
plt.tight_layout()

In [ ]:
# Dataframe

ai_gpa = (df.groupby('Primary_AI_Tool')['Improvement %'].mean().round(3) * 100).reset_index()
order = ai_gpa.sort_values('Improvement %', ascending=False)['Primary_AI_Tool'].tolist()

# Plot
fig, ax = plt.subplots(1, 1, figsize=(10, 4))
sns.barplot(data=ai_gpa, x='Primary_AI_Tool', y='Improvement %', order=order)
ax.set_title('GPA Improvement % by AI')
ax.set_xlabel('')

ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha='right')
for bar in ax.patches:
    height = bar.get_height()
    if height > 0:
        ax.text(bar.get_x() + bar.get_width() / 2, height + 0.02, f'{height:.2f}%', ha='center', va='bottom',
                fontsize=9)

sns.despine(left=False, right=True, top=True, bottom=False)
plt.tight_layout()

## **Hypothesis Tests**

#### Wilcoxon Test
    Non-parametric statistical test used to compare two related (paired) samples (Before and After)

##### **Hypotheses**

- **Null Hypothesis (H₀)**: The **median difference** between paired observations is **zero** (no change).
- **Alternative Hypothesis (H₁)**: The median difference **is not zero** (there is a change).

In [ ]:
GPA_test = df[['GPA_Baseline','GPA_Post_AI']]
GPA_test

from scipy.stats import wilcoxon

# Wilcoxon signed-rank test
stat, p = wilcoxon(GPA_test['GPA_Baseline'], GPA_test['GPA_Post_AI'])

print(f"Statistic = {stat:.4f}, p-value = {p:.4f}")

## **Insights**


#### **Statistics**:

* Analysis performed using synthetic data. Findings might not express real world situations. No strong correlations found among numeric features.
* Data shows GPA improvement post AI tools. The increase is not meaningful. It goes from 3.26 to 3.34.
* Copilot is the most used AI, followed by Gemini Pro and Perplexity. Differences between tools are small and the data is synthetic
* The main usage of the models goes to Exam Prep, followed by Literature Review and Essay Drafting.
* Fine arts is the field that is more concerned about ethics while Biology is the least.
* Software engineering is the field that shows the biggest GPA improvement after AI, followed by Biology and Business Admin.
* ChatGPT is the AI tool that brings more average improvement, followed by Perplexity and Copilot.

#### **Hypothesis Tests**:

* Both Hypothesis tests were successful to reject H0, which means that there's a valid change regarding GPA scores post AI usage.

#### **Thank You for taking the time to view this Notebook**!

If you found this analysis useful and have any feedback or suggestions, don't hesitate to contact me! We are here to learn!